In [2]:
import pandas as pd

#other_study_accession = "PRJEB50614"
#all_accessions = [study_accession, other_study_accession]
# Change for different studies
data_dir = "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/raw/metadata"

# ===== STEP 2: ENA DATA ON PROJECT NUMBER =====
# This section loads the ENA metadata and prepares it for updating

# STEP 2.1: Load the ENA metadata file
#Initial data release
ena_file = f"{data_dir}/ena_metadata_klebsiella_with_header_filtered.tsv"
ena_data_initial = pd.read_csv(ena_file, sep="\t", low_memory=False)  # Added low_memory=False to avoid warnings
#Incremental release
ena_file = f"{data_dir}/ena_metadata_klebsiella_with_header_filtered_r02_format.20240801.tsv"
ena_data_incremental = pd.read_csv(ena_file, sep="\t", low_memory=False)  # Added low_memory=False to avoid warnings

# Append together the incremental and initial data releases
ena_data = pd.concat([ena_data_initial, ena_data_incremental])

print(f"ENA data shape: {ena_data.shape}")
# print column names
print(f"ENA data column names: {ena_data.columns}")
display(ena_data.head(2))


ENA data shape: (89965, 118)
ENA data column names: Index(['sample_accession', 'run_accession', 'submission_accession',
       'assembly_software', 'library_selection', 'serotype',
       'environment_feature', 'last_updated', 'submitted_galaxy', 'germline',
       ...
       'taxonomic_classification', 'protocol_label', 'elevation', 'salinity',
       'sequencing_method', 'first_public', 'study_alias', 'ph', 'tissue_type',
       'isolation_source'],
      dtype='object', length=118)


,sample_accession,run_accession,submission_accession,assembly_software,library_selection,serotype,environment_feature,last_updated,submitted_galaxy,germline,...,taxonomic_classification,protocol_label,elevation,salinity,sequencing_method,first_public,study_alias,ph,tissue_type,isolation_source
0,SAMD00099771,DRR111584,DRA006333,NaN,RANDOM,NaN,NaN,2018-09-07,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2018-09-07,DRP003581,NaN,NaN,NaN
1,SAMD00099772,DRR111585,DRA006333,NaN,RANDOM,NaN,NaN,2018-09-07,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2018-09-07,DRP003581,NaN,NaN,NaN


In [8]:
# Find columns containing "ERR" in any of their values
def find_columns_with_pattern(df, pattern="ERR"):
    """Find which columns contain a specific pattern in their values."""
    matching_columns = {}
    
    for col in df.columns:
        # Check if column is string type or has string values
        if df[col].dtype == 'object' or df[col].dtype.name == 'string':
            # Count how many values contain the pattern
            matches = df[col].astype(str).str.contains(pattern, case=True, na=False).sum()
            if matches > 0:
                matching_columns[col] = matches
                # Show a few example values
                examples = df[df[col].astype(str).str.contains(pattern, case=True, na=False)][col].head(3).tolist()
                print(f"Column '{col}': {matches} values contain '{pattern}'")
                print(f"  Examples: {examples}\n")
    
    return matching_columns

# Usage:
matching_cols = find_columns_with_pattern(ena_data, "ERR")

Column 'run_accession': 43425 values contain 'ERR'
  Examples: ['ERR025527', 'ERR025569', 'ERR025567']

Column 'submitted_galaxy': 43425 values contain 'ERR'
  Examples: ['ftp.sra.ebi.ac.uk/vol1/run/ERR025/ERR025527/5193_7#1.srf', 'ftp.sra.ebi.ac.uk/vol1/run/ERR025/ERR025569/5197_7#12.srf', 'ftp.sra.ebi.ac.uk/vol1/run/ERR025/ERR025567/5197_7#10.srf']

Column 'submitted_ftp': 43425 values contain 'ERR'
  Examples: ['ftp.sra.ebi.ac.uk/vol1/run/ERR025/ERR025527/5193_7#1.srf', 'ftp.sra.ebi.ac.uk/vol1/run/ERR025/ERR025569/5197_7#12.srf', 'ftp.sra.ebi.ac.uk/vol1/run/ERR025/ERR025567/5197_7#10.srf']

Column 'sra_aspera': 3212 values contain 'ERR'
  Examples: ['fasp.sra.ebi.ac.uk:/vol1/err/ERR147/007/ERR1474977', 'fasp.sra.ebi.ac.uk:/vol1/err/ERR161/009/ERR1616339', 'fasp.sra.ebi.ac.uk:/vol1/err/ERR161/000/ERR1616360']

Column 'submitted_aspera': 43425 values contain 'ERR'
  Examples: ['fasp.sra.ebi.ac.uk:/vol1/run/ERR025/ERR025527/5193_7#1.srf', 'fasp.sra.ebi.ac.uk:/vol1/run/ERR025/ERR025569/

In [13]:
# Check the study_accession for run_accession ERR1008629
study_accession = ena_data[ena_data["run_accession"] == "SRR5082475"]["study_accession"].values[0]
print(f"Study accession for ERR1008629: {study_accession}")


Study accession for ERR1008629: PRJNA351909


In [ ]:
# Is there any column that contains 'age' in the column name?
age_cols = [col for col in ena_data.columns if "age" in col.lower()]
print(f"Columns containing 'age': {age_cols}")
# Print unique values of 'dev_stage' column      
print(ena_data["dev_stage"].unique())


Columns containing 'age': ['dev_stage']
[nan]


In [6]:
# Is there any column that contains 'host' in the column name?
host_cols = [col for col in ena_data.columns if "host" in col.lower()]
print(f"Columns containing 'host': {host_cols}")
# Print unique values of 'host' column      
print(ena_data["host_status"].unique())

Columns containing 'host': ['host_body_site', 'host_genotype', 'host_phenotype', 'host_tax_id', 'host_sex', 'host_growth_conditions', 'host_gravidity', 'host', 'host_status', 'submitted_host_sex']
[nan 'diseased' 'healthy' 'Carriage' 'carriage' 'preterm neonate'
 'Joint pain due to drained prosthetic joint' 'Sepsis' 'colonised'
 'critical patient' 'critical patient, multiplus antibioticals use'
 'not available' 'not provided' 'Not Known' 'Diseased' 'does not apply'
 'Infection' 'Colonization' 'Not known' 'disease' 'not collected'
 'Clinical' 'Screen' 'not known']
